# UMAP interactive map (light notebook)

Ce notebook utilise src/pokestats/umap_map.py pour garder la logique hors notebook.

In [ ]:
# Si besoin, decommenter
# %pip install -r requirements.txt

In [ ]:
from pathlib import Path
import sys
import plotly.express as px

ROOT = Path('.').resolve()
SRC_DIR = ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from pokestats.umap_map import build_umap_artifacts

In [ ]:
result = build_umap_artifacts(
    real_candidates=[
        ROOT / 'processed/pokemon_preprocessed_full.csv',
        ROOT / 'pokemon_data.csv',
    ],
    synthetic_csv=ROOT / 'artifacts/pokemon_synthetic.csv',
    artifacts_dir=ROOT / 'artifacts',
)

print('UMAP done')
print('-', result['html_type'])
print('-', result['html_gen'])
print('-', result['csv_projection'])
result['df_map'].head(3)

In [ ]:
result['fig_type'].show()
result['fig_gen'].show()

## Publication view with filters

Filtre interactif Type / Gen / Legendary pour la figure de rapport.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

pub_df = result['df_map'].copy()
pub_df['legendary_label'] = pub_df.get('is_legendary', 0).map({1: 'Legendary', 0: 'Non-Legendary'})

all_types = ['All'] + sorted(pub_df['type_1'].astype(str).unique().tolist())
all_gens = ['All'] + sorted([str(v) for v in pub_df['gen'].astype(int).unique().tolist()])
all_leg = ['All', 'Non-Legendary', 'Legendary']

w_type = widgets.Dropdown(options=all_types, value='All', description='Type:')
w_gen = widgets.Dropdown(options=all_gens, value='All', description='Gen:')
w_leg = widgets.Dropdown(options=all_leg, value='All', description='Legendary:')
out = widgets.Output()


def render_publication(_=None):
    with out:
        out.clear_output(wait=True)
        dff = pub_df.copy()
        if w_type.value != 'All':
            dff = dff[dff['type_1'].astype(str) == w_type.value]
        if w_gen.value != 'All':
            dff = dff[dff['gen'].astype(str) == w_gen.value]
        if w_leg.value != 'All':
            dff = dff[dff['legendary_label'] == w_leg.value]

        if dff.empty:
            print('No rows for selected filters.')
            return

        hover_cols = [c for c in ['name', 'type_1', 'type_2', 'gen', 'legendary_label', 'source', 'base_stats'] if c in dff.columns]
        fig_pub = px.scatter(
            dff,
            x='umap_1',
            y='umap_2',
            color='type_1',
            symbol='source',
            hover_data=hover_cols,
            opacity=0.82,
            title='Carte UMAP Pokemon - publication'
        )
        fig_pub.update_traces(marker={'size': 8, 'line': {'width': 0.3, 'color': 'white'}})
        fig_pub.update_layout(template='plotly_white', legend_title_text='Type / Source')
        fig_pub.show()

w_type.observe(render_publication, names='value')
w_gen.observe(render_publication, names='value')
w_leg.observe(render_publication, names='value')

display(widgets.HBox([w_type, w_gen, w_leg]))
display(out)
render_publication()

In [ ]:
pub_html = ROOT / 'artifacts/umap_pokemon_map_publication.html'
fig_export = px.scatter(
    pub_df,
    x='umap_1',
    y='umap_2',
    color='type_1',
    symbol='source',
    hover_data=[c for c in ['name', 'type_1', 'type_2', 'gen', 'legendary_label', 'source', 'base_stats'] if c in pub_df.columns],
    opacity=0.82,
    title='Carte UMAP Pokemon - publication (All)'
)
fig_export.update_traces(marker={'size': 8, 'line': {'width': 0.3, 'color': 'white'}})
fig_export.update_layout(template='plotly_white', legend_title_text='Type / Source')
fig_export.write_html(pub_html, include_plotlyjs='cdn')
print('Exported:', pub_html)